In [1]:
import subprocess
import sys

VLLM_PIN = "0.6.*"
BITSANDBYTES_PIN = "0.49.2"
AUTOAWQ_PIN = "0.2.*"
TRANSFORMERS_PIN = "4.46.*"
ACCELERATE_PIN = "1.1.*"
HTTPX_PIN = "0.27.*"
OPENAI_PIN = "1.54.*"

def pip_install(*specs):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *specs]
    print("installing:", " ".join(specs))
    subprocess.run(cmd, check=True)

pip_install(
    f"vllm=={VLLM_PIN}",
    f"transformers=={TRANSFORMERS_PIN}",
    f"accelerate=={ACCELERATE_PIN}",
    f"httpx=={HTTPX_PIN}",
    f"openai=={OPENAI_PIN}",
)

print("serving pins installed")

installing: vllm==0.6.* transformers==4.46.* accelerate==1.1.* httpx==0.27.* openai==1.54.*
serving pins installed


In [2]:
import os
import signal
import subprocess
import time
import urllib.request
import urllib.error

PORT = 8000
SERVER_LOG = "/content/server.log"

SERVER_ARGS = {
    "--model": "Qwen/Qwen2.5-1.5B-Instruct-AWQ",
    "--dtype": "half",
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": "8000",
    "--quantization": "awq",
    "--enable-auto-tool-choice": None,
    "--tool-call-parser": "hermes",
}

def build_cmd(args):
    cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server"]
    for key, value in args.items():
        if value is None:
            cmd.append(key)
        else:
            cmd += [key, str(value)]
    return cmd

def launch_server(args):
    cmd = build_cmd(args)
    print("launching:", " ".join(cmd))
    logf = open(SERVER_LOG, "wb")
    proc = subprocess.Popen(
        cmd,
        stdout=logf,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    print(f"server pid {proc.pid}, logging to {SERVER_LOG}")
    return proc

def tail_log(path=SERVER_LOG, n=30):
    try:
        with open(path, "r", errors="replace") as fh:
            return "".join(fh.readlines()[-n:])
    except FileNotFoundError:
        return "(no log file yet)"

def wait_for_health(timeout_s=300, interval_s=3):
    url = f"http://localhost:{PORT}/v1/models"
    deadline = time.time() + timeout_s

    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as response:
                if response.status == 200:
                    waited = int(timeout_s - (deadline - time.time()))
                    print(
                        f"server healthy after about {waited}s: "
                        f"{url} -> 200"
                    )
                    return True
        except (urllib.error.URLError, ConnectionError, OSError):
            pass

        time.sleep(interval_s)

    print(f"TIMED OUT after {timeout_s}s")
    print(tail_log())
    return False

server = launch_server(SERVER_ARGS)
healthy = wait_for_health()

launching: /usr/bin/python3 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-1.5B-Instruct-AWQ --dtype half --max-model-len 4096 --gpu-memory-utilization 0.85 --port 8000 --quantization awq --enable-auto-tool-choice --tool-call-parser hermes
server pid 3714, logging to /content/server.log
server healthy after about 135s: http://localhost:8000/v1/models -> 200


In [4]:
!python bench.py \
  --base-url http://localhost:8000 \
  --model "Qwen/Qwen2.5-1.5B-Instruct-AWQ" \
  --concurrency 1,2,4,8,16 \
  --requests-per-level 20 \
  --prompt-file prompts.txt \
  --out bench_report.json

[level 1] tok/s=78.92 ttft_p95=0.1344 errors=0
[level 2] tok/s=164.19 ttft_p95=0.0877 errors=0
[level 4] tok/s=248.33 ttft_p95=0.1989 errors=0
[level 8] tok/s=476.07 ttft_p95=0.2019 errors=0
[level 16] tok/s=679.42 ttft_p95=0.2421 errors=0

conc     tok/s   ttft_p50   ttft_p95   lat_p95    ok   err
----------------------------------------------------------
   1     78.92      0.079      0.134     1.655    20     0
   2    164.19      0.051      0.088     1.547    20     0
   4    248.33      0.084      0.199     2.472    20     0
   8    476.07      0.140      0.202     1.957    20     0
  16    679.42      0.238      0.242     2.413    20     0

wrote bench_report.json (run appended)


In [5]:
import json

levels = json.load(open("bench_report.json"))["runs"][-1]["levels"]

for L in levels:
    print(
        f"c={L['concurrency']:>2}  "
        f"tok/s={L['tokens_per_s']:>7.1f}  "
        f"ttft_p95={L['ttft_p95_s']:.3f}  "
        f"lat_p95={L['latency_p95_s']:.3f}  "
        f"errors={L['errors']}"
    )

TARGET_P95_S = 3.0

under = [
    L for L in levels
    if L["latency_p95_s"] <= TARGET_P95_S
]

knee = (
    max(under, key=lambda L: L["concurrency"])
    if under else None
)

print("target p95:", TARGET_P95_S)
print("knee:", knee)

c= 1  tok/s=   78.9  ttft_p95=0.134  lat_p95=1.655  errors=0
c= 2  tok/s=  164.2  ttft_p95=0.088  lat_p95=1.547  errors=0
c= 4  tok/s=  248.3  ttft_p95=0.199  lat_p95=2.472  errors=0
c= 8  tok/s=  476.1  ttft_p95=0.202  lat_p95=1.957  errors=0
c=16  tok/s=  679.4  ttft_p95=0.242  lat_p95=2.413  errors=0
target p95: 3.0
knee: {'concurrency': 16, 'tokens_per_s': 679.42, 'ttft_p50_s': 0.2385, 'ttft_p95_s': 0.2421, 'latency_p95_s': 2.4134, 'errors': 0, 'ok': 20, 'wall_s': 3.056}


In [8]:
import json

capacity_note = """# Capacity note (team, one page)

## The numbers

- Locked model: `Qwen/Qwen2.5-1.5B-Instruct-AWQ`
- Target p95 end-to-end latency (your SLO today): `3.0` seconds
- Knee concurrency (highest concurrency whose p95 is still under target): `16` (sweep-bounded)
- Tokens per second at the knee: `679.42`
- Max sustainable request rate at the target p95: `6.54 req/s`

## The limiting family

- Memory-bound: doubling concurrency from 8 to 16 increased throughput by only 43% while p95 rose from 1.957 to 2.413 seconds, indicating that decode is approaching the memory-bandwidth ceiling.

## Why the knee, not the peak

- The knee represents capacity that still meets the 3-second p95 SLO, while throughput beyond that latency target would not be a reliable service commitment.
"""

with open("capacity-note.md", "w") as f:
    f.write(capacity_note)

with open("knee.json", "w") as f:
    json.dump(
        {
            "target_p95_s": TARGET_P95_S,
            "knee_concurrency": knee["concurrency"] if knee else None,
        },
        f,
        indent=2,
    )

print(capacity_note)
print("\nknee.json:")
print(json.dumps({
    "target_p95_s": TARGET_P95_S,
    "knee_concurrency": knee["concurrency"] if knee else None,
}, indent=2))

# Capacity note (team, one page)

## The numbers

- Locked model: `Qwen/Qwen2.5-1.5B-Instruct-AWQ`
- Target p95 end-to-end latency (your SLO today): `3.0` seconds
- Knee concurrency (highest concurrency whose p95 is still under target): `16` (sweep-bounded)
- Tokens per second at the knee: `679.42`
- Max sustainable request rate at the target p95: `6.54 req/s`

## The limiting family

- Memory-bound: doubling concurrency from 8 to 16 increased throughput by only 43% while p95 rose from 1.957 to 2.413 seconds, indicating that decode is approaching the memory-bandwidth ceiling.

## Why the knee, not the peak

- The knee represents capacity that still meets the 3-second p95 SLO, while throughput beyond that latency target would not be a reliable service commitment.


knee.json:
{
  "target_p95_s": 3.0,
  "knee_concurrency": 16
}


In [9]:
def shutdown_server(proc=None, port=PORT):
    try:
        proc = server if proc is None else proc
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        print(f"sent SIGTERM to process group of pid {proc.pid}")
    except (ProcessLookupError, NameError):
        print("no server process to kill")

    time.sleep(3)

    try:
        with urllib.request.urlopen(
            f"http://localhost:{port}/v1/models",
            timeout=2,
        ):
            print(f"WARNING: port {port} still answering")
    except (urllib.error.URLError, ConnectionError, OSError):
        print(f"port {port} is free")

shutdown_server()

sent SIGTERM to process group of pid 3714
port 8000 is free


In [10]:
# Green-check verifier for Lab W3D5 (benchmark harness).
# Paste this as the last cell of your day-5 notebook and run it. It reads
# bench_report.json (from the harness) and capacity-note.md, and checks the
# schema, that at least four concurrency levels ran, that errors are zero or
# explained, and that the capacity note is filled in.
#
# Last line is exactly one of:
#   GREEN CHECK: PASS
#   GREEN CHECK: FAIL (<reason>)
# No interactivity, no arguments; exit code matches.

import json, os, re

LEVEL_KEYS = {"concurrency", "tokens_per_s", "ttft_p50_s", "ttft_p95_s",
              "latency_p95_s", "errors"}


class _Stop(Exception):
    """Ends the check without killing the notebook kernel."""


def fail(reason: str) -> "NoReturn":
    print(f"GREEN CHECK: FAIL ({reason})")
    raise _Stop()


def main() -> None:
    # 1) bench report
    if not os.path.exists("bench_report.json"):
        fail("bench_report.json not found; run the harness in Cell 3")
    try:
        with open("bench_report.json") as fh:
            document = json.load(fh)
    except json.JSONDecodeError as exc:
        fail(f"bench_report.json is not valid JSON: {exc}")

    # bench.py appends each sweep to a "runs" list rather than overwriting, so
    # the file is a document and the thing to grade is the most recent run. A
    # bare list is also accepted, for a report assembled by hand.
    if isinstance(document, dict) and isinstance(document.get("runs"), list):
        if not document["runs"]:
            fail("bench_report.json has no runs; the harness wrote nothing")
        levels = document["runs"][-1].get("levels")
        if not isinstance(levels, list):
            fail("the most recent run in bench_report.json has no levels list")
    elif isinstance(document, list):
        levels = document
    else:
        fail("bench_report.json must be the harness output ({'runs': [...]}) "
             "or a bare list of per-level objects")
    if len(levels) < 4:
        fail(f"need at least 4 concurrency levels, found {len(levels)}")

    total_errors = 0
    for i, L in enumerate(levels):
        if not isinstance(L, dict):
            fail(f"level {i} is not an object")
        missing = LEVEL_KEYS - set(L)
        if missing:
            fail(f"level {i} missing keys: {sorted(missing)}")
        if not isinstance(L["errors"], int) or L["errors"] < 0:
            fail(f"level {i} errors must be a non-negative integer")
        total_errors += L["errors"]

    # 2) the knee file from Cell 5
    if not os.path.exists("knee.json"):
        fail("knee.json not found; write it in Cell 5")
    try:
        with open("knee.json") as fh:
            knee = json.load(fh)
    except json.JSONDecodeError as exc:
        fail(f"knee.json is not valid JSON: {exc}")
    target = knee.get("target_p95_s")
    if not isinstance(target, (int, float)) or target <= 0:
        fail("target_p95_s is not a positive number; set TARGET_P95_S to your "
             "real SLO before computing the knee (the 'target left at zero' "
             "failure mode)")
    kc = knee.get("knee_concurrency")
    if not isinstance(kc, int) or kc < 1:
        fail("knee_concurrency is empty: no level stayed under your target. "
             "Either your SLO is stricter than this stack can serve (explain "
             "that in the note) or the target was never set from the card")

    # errors must be zero, OR explained in the capacity note
    # 3) capacity note filled in
    if not os.path.exists("capacity-note.md"):
        fail("capacity-note.md not found")
    with open("capacity-note.md") as fh:
        note = fh.read()
    remaining = re.findall(r"FILL:", note)
    if remaining:
        fail(f"capacity-note.md has {len(remaining)} unfilled FILL: placeholders")

    if total_errors > 0 and not re.search(r"error", note, re.I):
        fail(f"{total_errors} request errors in the sweep and no explanation in "
             "capacity-note.md; zero errors, or explain them")

    # sanity: throughput should be present and positive somewhere
    if not any(isinstance(L["tokens_per_s"], (int, float)) and L["tokens_per_s"] > 0
               for L in levels):
        fail("no level reports positive tokens_per_s")

    concurrencies = sorted(L["concurrency"] for L in levels)
    print(f"levels: {len(levels)}, concurrencies: {concurrencies}, "
          f"total errors: {total_errors}")
    print("capacity-note.md: all fields filled")
    print("GREEN CHECK: PASS")


try:
    main()
except _Stop:
    # A notebook cell cannot exit nonzero without printing a red traceback over
    # the result line, so only signal by exit code when run as a plain script.
    try:
        get_ipython()  # defined only inside IPython/Colab
    except NameError:
        raise SystemExit(1)

levels: 5, concurrencies: [1, 2, 4, 8, 16], total errors: 0
capacity-note.md: all fields filled
GREEN CHECK: PASS


In [12]:
from google.colab import files

for f_ in ["bench_report.json", "capacity-note.md"]:
    files.download(f_)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
import json

# Step 1: load your own benchmark results
try:
    levels = json.load(open("bench_report.json"))["runs"][-1]["levels"]
except FileNotFoundError:
    levels = [
        {"concurrency": 1,  "tokens_per_s": 38.2,  "latency_p95_s": 0.9, "errors": 0},
        {"concurrency": 2,  "tokens_per_s": 71.5,  "latency_p95_s": 1.1, "errors": 0},
        {"concurrency": 4,  "tokens_per_s": 128.4, "latency_p95_s": 1.4, "errors": 0},
        {"concurrency": 8,  "tokens_per_s": 210.7, "latency_p95_s": 2.3, "errors": 0},
        {"concurrency": 16, "tokens_per_s": 224.9, "latency_p95_s": 5.8, "errors": 0},
    ]
    print("using the sample bench_report -- swap in your own file for a real answer")

# Step 2: calculate cost per million output tokens
def cost_per_million_tokens(tokens_per_s, gpu_hourly_usd):
    tokens_per_hour = tokens_per_s * 3600
    million_tokens_per_hour = tokens_per_hour / 1_000_000
    return round(gpu_hourly_usd / million_tokens_per_hour, 4)

GPU_HOURLY_USD = 0.35

for L in levels:
    L["cost_per_million_tokens_usd"] = cost_per_million_tokens(
        L["tokens_per_s"],
        GPU_HOURLY_USD,
    )

for L in levels:
    print(
        f"c={L['concurrency']:>2}  "
        f"tok/s={L['tokens_per_s']:>7.1f}  "
        f"p95={L['latency_p95_s']:.2f}s  "
        f"$/M tok=${L['cost_per_million_tokens_usd']}"
    )

# Step 3: find and price the knee
TARGET_P95_S = 3.0

under_target = [
    L for L in levels
    if L["latency_p95_s"] <= TARGET_P95_S
]

knee = (
    max(under_target, key=lambda L: L["concurrency"])
    if under_target else None
)

print("\nknee:", knee)

past_knee = [
    L for L in levels
    if knee and L["concurrency"] > knee["concurrency"]
]

if past_knee:
    cheapest_past_knee = min(
        past_knee,
        key=lambda L: L["cost_per_million_tokens_usd"],
    )
    print(
        "cheapest $/M token level past the knee "
        "(SLO-violating):",
        cheapest_past_knee,
    )
    print(
        "-> cheaper on paper, but its p95 already exceeds "
        "your SLO -- not real usable capacity at your target."
    )
else:
    print("No measured level is past the knee; the knee is sweep-bounded.")

c= 1  tok/s=   78.9  p95=1.66s  $/M tok=$1.2319
c= 2  tok/s=  164.2  p95=1.55s  $/M tok=$0.5921
c= 4  tok/s=  248.3  p95=2.47s  $/M tok=$0.3915
c= 8  tok/s=  476.1  p95=1.96s  $/M tok=$0.2042
c=16  tok/s=  679.4  p95=2.41s  $/M tok=$0.1431

knee: {'concurrency': 16, 'tokens_per_s': 679.42, 'ttft_p50_s': 0.2385, 'ttft_p95_s': 0.2421, 'latency_p95_s': 2.4134, 'errors': 0, 'ok': 20, 'wall_s': 3.056, 'cost_per_million_tokens_usd': 0.1431}
No measured level is past the knee; the knee is sweep-bounded.


In [14]:
import math
import json

# Step 4: calculate the replica plan
def replicas_needed(required_tokens_per_s, knee_tokens_per_s):
    return math.ceil(required_tokens_per_s / knee_tokens_per_s)

def scale_out_cost(required_tokens_per_s, knee, gpu_hourly_usd):
    n = replicas_needed(
        required_tokens_per_s,
        knee["tokens_per_s"],
    )
    return {
        "required_tokens_per_s": required_tokens_per_s,
        "replicas_needed": n,
        "total_hourly_cost_usd": round(n * gpu_hourly_usd, 2),
        "effective_p95_s": knee["latency_p95_s"],
    }

targets = [
    knee["tokens_per_s"] * multiplier
    for multiplier in (1.0, 1.5, 2.0, 3.0)
]

scale_plan = [
    scale_out_cost(target, knee, GPU_HOURLY_USD)
    for target in targets
]

for row in scale_plan:
    print(row)

# Step 5: write the final cost report
report = {
    "gpu_hourly_usd": GPU_HOURLY_USD,
    "target_p95_s": TARGET_P95_S,
    "levels": levels,
    "knee": knee,
    "scale_out_plan": scale_plan,
}

with open("cost_report.json", "w") as f:
    json.dump(report, f, indent=2)

print("\ncost_report.json:")
print(json.dumps(report, indent=2))

{'required_tokens_per_s': 679.42, 'replicas_needed': 1, 'total_hourly_cost_usd': 0.35, 'effective_p95_s': 2.4134}
{'required_tokens_per_s': 1019.1299999999999, 'replicas_needed': 2, 'total_hourly_cost_usd': 0.7, 'effective_p95_s': 2.4134}
{'required_tokens_per_s': 1358.84, 'replicas_needed': 2, 'total_hourly_cost_usd': 0.7, 'effective_p95_s': 2.4134}
{'required_tokens_per_s': 2038.2599999999998, 'replicas_needed': 3, 'total_hourly_cost_usd': 1.05, 'effective_p95_s': 2.4134}

cost_report.json:
{
  "gpu_hourly_usd": 0.35,
  "target_p95_s": 3.0,
  "levels": [
    {
      "concurrency": 1,
      "tokens_per_s": 78.92,
      "ttft_p50_s": 0.079,
      "ttft_p95_s": 0.1344,
      "latency_p95_s": 1.6555,
      "errors": 0,
      "ok": 20,
      "wall_s": 25.381,
      "cost_per_million_tokens_usd": 1.2319
    },
    {
      "concurrency": 2,
      "tokens_per_s": 164.19,
      "ttft_p50_s": 0.051,
      "ttft_p95_s": 0.0877,
      "latency_p95_s": 1.5471,
      "errors": 0,
      "ok": 20,
 

In [15]:
#!/usr/bin/env python3
# Green check for the extra W3D5 lab (cost per million tokens, scale-out).
# Run next to cost_report.json:  python verify.py
# Prints exactly one line last: GREEN CHECK: PASS  or  GREEN CHECK: FAIL (<reason>)
# stdlib only.
#
# The lab is pure arithmetic over the student's own bench levels, so this
# recomputes EVERYTHING from the levels in the report: per-level cost, knee
# selection, and the whole scale-out plan. It works identically for the sample
# bench and a student's real one.
import json, math, os
from typing import NoReturn


class _Stop(Exception):
    pass


def _fail(reason) -> NoReturn:
    print("GREEN CHECK: FAIL (%s)" % reason)
    raise _Stop()


def main():
    if not os.path.isfile("cost_report.json"):
        _fail("cost_report.json not found; run Step 5 first")
    try:
        with open("cost_report.json") as f:
            r = json.load(f)
    except json.JSONDecodeError as e:
        _fail("cost_report.json is not valid JSON: %s" % e)

    for key in ("gpu_hourly_usd", "target_p95_s", "levels", "knee", "scale_out_plan"):
        if key not in r:
            _fail("missing key '%s'" % key)
    rate, slo = r["gpu_hourly_usd"], r["target_p95_s"]
    if not isinstance(rate, (int, float)) or rate <= 0:
        _fail("gpu_hourly_usd must be a positive dollars-per-hour figure")
    if not isinstance(slo, (int, float)) or slo <= 0:
        _fail("target_p95_s must be a positive SLO in seconds")

    levels = r["levels"]
    if not isinstance(levels, list) or len(levels) < 3:
        _fail("levels must hold the bench sweep (at least 3 concurrency levels)")
    for L in levels:
        for f_ in ("concurrency", "tokens_per_s", "latency_p95_s",
                   "cost_per_million_tokens_usd"):
            if not isinstance(L.get(f_), (int, float)):
                _fail("level %r lacks numeric %s" % (L.get("concurrency"), f_))
        if not L["tokens_per_s"] or L["tokens_per_s"] <= 0:
            _fail("level %s reports tokens_per_s <= 0 (an all-error level); rerun the sweep" % L.get("concurrency"))
        want_cost = round(rate / (L["tokens_per_s"] * 3600 / 1_000_000), 4)
        if abs(L["cost_per_million_tokens_usd"] - want_cost) > max(0.0002, want_cost * 0.01):
            _fail("concurrency %s: cost %.4f, the formula gives %.4f "
                  "(tokens/s vs tokens/hour, or a non-hourly rate?)"
                  % (L["concurrency"], L["cost_per_million_tokens_usd"], want_cost))

    under = [L for L in levels if L["latency_p95_s"] <= slo]
    if not under:
        _fail("no level sits under the SLO, so no knee exists; the report "
              "should not have gotten this far (see failure modes)")
    want_knee = max(under, key=lambda L: L["concurrency"])
    knee = r["knee"]
    if not isinstance(knee, dict) or knee.get("concurrency") != want_knee["concurrency"]:
        _fail("knee is concurrency %s; the largest level under the %.1fs SLO "
              "is concurrency %s" % ((knee or {}).get("concurrency"), slo,
                                     want_knee["concurrency"]))

    plan = r["scale_out_plan"]
    want_targets = [round(want_knee["tokens_per_s"] * m, 6) for m in (1.0, 1.5, 2.0, 3.0)]
    if not isinstance(plan, list) or len(plan) != 4:
        _fail("scale_out_plan must hold the four multiples 1.0, 1.5, 2.0, 3.0")
    for row, want_req in zip(plan, want_targets):
        req = row.get("required_tokens_per_s")
        if not isinstance(req, (int, float)) or abs(req - want_req) > max(0.5, want_req * 0.01):
            _fail("plan targets must be the knee's throughput x (1, 1.5, 2, 3); "
                  "got %r, expected %.1f" % (req, want_req))
        want_n = math.ceil(want_req / want_knee["tokens_per_s"] - 1e-9)
        if row.get("replicas_needed") != want_n:
            _fail("required %.1f tok/s: replicas_needed=%r, ceil gives %d"
                  % (req, row.get("replicas_needed"), want_n))
        want_cost = round(want_n * rate, 2)
        if abs(row.get("total_hourly_cost_usd", 1e9) - want_cost) > 0.011:
            _fail("required %.1f tok/s: hourly cost %r, %d replicas at %.2f/h "
                  "gives %.2f" % (req, row.get("total_hourly_cost_usd"),
                                  want_n, rate, want_cost))
        if abs(row.get("effective_p95_s", 1e9) - want_knee["latency_p95_s"]) > 0.011:
            _fail("effective_p95_s must stay at the knee's p95: replicas run at "
                  "the safe concurrency, that is the whole model")

    print("recomputed costs, knee and scale-out plan all agree")
    print("GREEN CHECK: PASS")


if __name__ == "__main__":
    try:
        main()
    except _Stop:
        raise SystemExit(1)


recomputed costs, knee and scale-out plan all agree
GREEN CHECK: PASS


In [16]:
from google.colab import files

for filename in [
    "bench_report.json",
    "capacity-note.md",
    "knee.json",
    "cost_report.json",
]:
    files.download(filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>